# Metadata extraction/filtering

In certain scenarios, RAG benefits from filtering database queries using metadata. This is one of those scenarios. The knowledge base is annual reports from Microsoft, Alphabet, and Meta for the years 2020-2023. We'll chunk the reports, insert the chunks into a ChromaDB database, and tag each chunk with metadata indicating the company name and the year of the report. When asked a question such as "What was Microsoft's revenue in 2022," the app will query the database for relevant chunks where company equals "microsoft" and year equals "2022." This way, the app can provide accurate answers without mixing up annual reports from different companies and different years. We'll use an LLM to determine what metadata values, if any, to use for filtering.

Start by creating a ChromaDB database in the "chroma" subdirectory and creating a collection named "Annual_Reports:"

In [1]:
import chromadb

client = chromadb.PersistentClient('chroma')
collection = client.get_or_create_collection(name='Annual_Reports')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Now use LLamaIndex's `DocxReader` and `PdfReader` classes to extract text from the annual reports in the "Data" subdirectory and LlamaIndex's `SentenceSplitter` class to chunk the text. Then insert the chunks into the database along with metadata identifying the company and the year:

In [2]:
import os, re
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file.docs import DocxReader, PDFReader

path = 'Data'
pattern = re.compile(r"^(?P<company>\w+)_(?P<year>\d{4})\.(?P<extension>\w+)$")

for filename in os.listdir(path):
    match = pattern.match(filename)

    if match:
        company = match.group("company").lower()
        year = match.group("year").lower()
        extension = match.group("extension").lower()
        full_path = os.path.join(path, filename)

        if extension == 'pdf':
            reader = PDFReader()
        elif extension == 'docx':
            reader = DocxReader()
        else:
            continue

        print(f'Processing {filename}...')
        document = reader.load_data(full_path)
        splitter = SentenceSplitter(chunk_size=512, chunk_overlap=40)
        nodes = splitter.get_nodes_from_documents(document)

        for i, node in enumerate(nodes):
            if len(node.text) > 128:
                collection.add(
                    documents=[node.text],
                    metadatas=[{ 'company': company, 'year': year }],
                    ids=[f'{company}-{year}-{i:05}']
                )

Processing Alphabet_2020.pdf...


2026-08-19 23:36:32,243 - WARNING - Add of existing embedding ID: microsoft-2023-00116
2026-08-19 23:36:32,245 - WARNING - Add of existing embedding ID: microsoft-2023-00117
2026-08-19 23:36:32,274 - WARNING - Add of existing embedding ID: microsoft-2023-00118
2026-08-19 23:36:32,277 - WARNING - Add of existing embedding ID: microsoft-2023-00119
2026-08-19 23:36:32,279 - WARNING - Add of existing embedding ID: microsoft-2023-00120
2026-08-19 23:36:32,281 - WARNING - Add of existing embedding ID: microsoft-2023-00121
2026-08-19 23:36:32,284 - WARNING - Add of existing embedding ID: microsoft-2023-00122
2026-08-19 23:36:32,288 - WARNING - Add of existing embedding ID: microsoft-2023-00123
2026-08-19 23:36:32,289 - WARNING - Add of existing embedding ID: microsoft-2023-00124
2026-08-19 23:36:32,289 - WARNING - Add of existing embedding ID: microsoft-2023-00125
2026-08-19 23:36:32,291 - WARNING - Add of existing embedding ID: microsoft-2023-00126
2026-08-19 23:36:32,294 - WARNING - Add of 

Processing Alphabet_2021.pdf...


2026-08-19 23:37:10,159 - WARNING - Insert of existing embedding ID: alphabet-2021-00002
2026-08-19 23:37:10,160 - WARNING - Add of existing embedding ID: alphabet-2021-00002
2026-08-19 23:37:10,221 - WARNING - Insert of existing embedding ID: alphabet-2021-00003
2026-08-19 23:37:10,227 - WARNING - Add of existing embedding ID: alphabet-2021-00003
2026-08-19 23:37:10,305 - WARNING - Insert of existing embedding ID: alphabet-2021-00004
2026-08-19 23:37:10,309 - WARNING - Add of existing embedding ID: alphabet-2021-00004
2026-08-19 23:37:10,381 - WARNING - Insert of existing embedding ID: alphabet-2021-00005
2026-08-19 23:37:10,395 - WARNING - Add of existing embedding ID: alphabet-2021-00005
2026-08-19 23:37:10,496 - WARNING - Insert of existing embedding ID: alphabet-2021-00006
2026-08-19 23:37:10,498 - WARNING - Add of existing embedding ID: alphabet-2021-00006
2026-08-19 23:37:10,572 - WARNING - Insert of existing embedding ID: alphabet-2021-00007
2026-08-19 23:37:10,572 - WARNING - 

Processing Alphabet_2022.pdf...


...

2026-08-19 23:43:17,540 - WARNING - Insert of existing embedding ID: microsoft-2021-00000
2026-08-19 23:43:17,542 - WARNING - Add of existing embedding ID: microsoft-2021-00000
2026-08-19 23:43:17,587 - WARNING - Insert of existing embedding ID: microsoft-2021-00001
2026-08-19 23:43:17,587 - WARNING - Add of existing embedding ID: microsoft-2021-00001
2026-08-19 23:43:17,657 - WARNING - Insert of existing embedding ID: microsoft-2021-00002
2026-08-19 23:43:17,657 - WARNING - Add of existing embedding ID: microsoft-2021-00002
2026-08-19 23:43:17,751 - WARNING - Insert of existing embedding ID: microsoft-2021-00003
2026-08-19 23:43:17,751 - WARNING - Add of existing embedding ID: microsoft-2021-00003
2026-08-19 23:43:17,839 - WARNING - Insert of existing embedding ID: microsoft-2021-00004
2026-08-19 23:43:17,839 - WARNING - Add of existing embedding ID: microsoft-2021-00004
2026-08-19 23:43:17,935 - WARNING - Insert of existing embedding ID: microsoft-2021-00005
2026-08-19 23:43:17,935 -

Processing Microsoft_2022.docx...


2026-08-19 23:43:38,252 - WARNING - Insert of existing embedding ID: microsoft-2022-00000
2026-08-19 23:43:38,254 - WARNING - Add of existing embedding ID: microsoft-2022-00000
2026-08-19 23:43:38,303 - WARNING - Insert of existing embedding ID: microsoft-2022-00001
2026-08-19 23:43:38,303 - WARNING - Add of existing embedding ID: microsoft-2022-00001
2026-08-19 23:43:38,416 - WARNING - Insert of existing embedding ID: microsoft-2022-00002
2026-08-19 23:43:38,416 - WARNING - Add of existing embedding ID: microsoft-2022-00002
2026-08-19 23:43:38,489 - WARNING - Insert of existing embedding ID: microsoft-2022-00003
2026-08-19 23:43:38,491 - WARNING - Add of existing embedding ID: microsoft-2022-00003
2026-08-19 23:43:38,588 - WARNING - Insert of existing embedding ID: microsoft-2022-00004
2026-08-19 23:43:38,592 - WARNING - Add of existing embedding ID: microsoft-2022-00004
2026-08-19 23:43:38,705 - WARNING - Insert of existing embedding ID: microsoft-2022-00005
2026-08-19 23:43:38,707 -

Processing Microsoft_2023.docx...


2026-08-19 23:43:59,738 - WARNING - Insert of existing embedding ID: microsoft-2023-00000
2026-08-19 23:43:59,739 - WARNING - Add of existing embedding ID: microsoft-2023-00000
2026-08-19 23:43:59,855 - WARNING - Insert of existing embedding ID: microsoft-2023-00001
2026-08-19 23:43:59,855 - WARNING - Add of existing embedding ID: microsoft-2023-00001
2026-08-19 23:43:59,980 - WARNING - Insert of existing embedding ID: microsoft-2023-00002
2026-08-19 23:43:59,987 - WARNING - Add of existing embedding ID: microsoft-2023-00002
2026-08-19 23:44:00,098 - WARNING - Insert of existing embedding ID: microsoft-2023-00003
2026-08-19 23:44:00,100 - WARNING - Add of existing embedding ID: microsoft-2023-00003
2026-08-19 23:44:00,216 - WARNING - Insert of existing embedding ID: microsoft-2023-00004
2026-08-19 23:44:00,220 - WARNING - Add of existing embedding ID: microsoft-2023-00004
2026-08-19 23:44:00,350 - WARNING - Insert of existing embedding ID: microsoft-2023-00005
2026-08-19 23:44:00,361 -

Define a function to extract metadata values from a question and return a JSON structure that can be used in a `where` clause in a ChromaDB query:

In [3]:
from openai import OpenAI

def extract_metadata(question):
    content = f'''
        The question below will be used to query a vector database. Each item in the database
        has metadata values named "company" and "year" designating the company that the item
        pertains to and the year it was recorded. From the question, generate JSON that
        indicates which, if any, metadata values should be included in the query and what
        values to assign to them. If the company name is "Google," use "Alphabet" instead.
        If the company name is "Facebook," use "Meta" instead.
        
        If a company name is detected but a year is not, use this format:
    
        {{
            "company": "value"
        }}

        If a year is detected but a company name is not, use this format:
    
        {{
            "year": "value"
        }}

        If both a company name and a year are detected, use this format:

        {{
            "$and": [
                "company": "value",
                "year": "value"
            ]
        }}

        Do not return markdown. Only return JSON. All JSON values must be strings.
        
        Question:
        {question}
        '''

    messages = [{ 'role': 'user', 'content': content }]
    client = OpenAI()

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        response_format={ 'type': 'json_object' }
    )

    return response.choices[0].message.content

Test the function with a question that contains two metadata values:

In [4]:
print(extract_metadata("What was Microsoft's revenue in 2022?"))

2026-08-19 23:44:21,924 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{
    "$and": [
        {
            "company": "Microsoft"
        },
        {
            "year": "2022"
        }
    ]
}


Test it with a question that contains one metadata value:

In [5]:
print(extract_metadata("Who is Google's CEO?"))

2026-08-19 23:44:23,566 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{
    "company": "Alphabet"
}


Test it with a question that contains no metadata values:

In [6]:
print(extract_metadata('What is sustainability?'))

2026-08-19 23:44:25,000 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{}


Load `jina-reranker-v1-turbo-en` for reranking query results:

In [9]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', trust_remote_code=True)

2026-08-19 23:47:02,736 - INFO - No device provided, using cpu
2026-08-19 23:47:02,923 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-19 23:47:03,022 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-08-19 23:47:03,035 - INFO - No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
2026-08-19 23:47:03,137 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
2026-08-19 23:47:03,294 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-08-19 23:47:03,407 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2

Define a function for asking questions about the annual reports:

In [20]:
import json
from openai import OpenAI

def answer_question(question):
    client = chromadb.PersistentClient('chroma')
    collection = client.get_collection(name='Annual_Reports')    

    # 1. Extract metadata from question
    metadata_raw = extract_metadata(question).strip().lower()

    # 2. Smart parser: Convert string format into a valid dictionary
    where_filter = {}
    if ":" in metadata_raw and not metadata_raw.startswith("{"):
        # Handles plain strings like "company: microsoft"
        key, value = metadata_raw.split(":", 1)
        where_filter = {key.strip(): value.strip()}
    else:
        try:
            # Handles valid JSON strings
            where_filter = json.loads(metadata_raw)
            
            # --- FIXED AUTO-UNPACK FOR CHROMADB NESTING ISSUE ---
            for op in ["$and", "$or"]:
                if op in where_filter and isinstance(where_filter[op], list):
                    expr_list = where_filter[op]
                    # If the list only has 1 item, extract that item directly
                    if len(expr_list) == 1:
                        where_filter = expr_list[0]
                        break
                    # If an $and contains a single dict with multiple keys, split them out
                    elif len(expr_list) == 2 and isinstance(expr_list[0], dict) and len(expr_list[0]) > 1:
                        # Fallback: flatten the inner dict completely
                        where_filter = expr_list[0]
                        break
            
        except json.JSONDecodeError:
            print(f"Warning: Could not parse metadata string: {metadata_raw}")
            where_filter = {}
            
        except json.JSONDecodeError:
            print(f"Warning: Could not parse metadata string: {metadata_raw}")
            where_filter = {}

    # 3. Query the database using the true dictionary
    results = collection.query(
        query_texts=[question],
        where=where_filter,
        n_results=40
    )
    
    # 4. Process results and pass to LLM
    documents = results['documents'][0]

    if len(documents) == 0:
        print("I don't know.")
        return
        
    # Rerank the results
    ranked_documents = model.rank(question, documents, return_documents=True, top_k=20)
    context = '\n\n'.join(x['text'] for x in ranked_documents)

    # Submit the question and the chunks to an LLM and stream the response
    openai_client = OpenAI()

    content = f'''
        Answer the following question using the provided context, and if the
        answer is not contained within the context, say "I don't know." Explain
        your answer if possible. Do not mention the provided context in your
        output. Do not use markdown formatting. Do not use backslash characters.
        
        Question:
        {question}

        Context:
        {context}
        '''

    messages = [{ 'role': 'user', 'content': content }]

    response = openai_client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        stream=True
    )

    for chunk in response:
        content = chunk.choices[0].delta.content
        if content is not None:
            print(content, end='')


Ask a question regarding Microsoft's 2022 revenue:

In [21]:
answer_question("What was Microsoft's revenue in 2022?")

2026-08-19 23:56:22,353 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-08-19 23:56:23,586 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 2/2 [00:12<00:00,  6.02s/it]
2026-08-19 23:56:36,745 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Microsoft's revenue in 2022 was $198.27 billion.

Do the same for Google in 2023:

In [22]:
answer_question("What was Google's revenue in 2023?")

2026-08-19 23:56:43,580 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-08-19 23:56:44,545 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 2/2 [00:11<00:00,  5.90s/it]
2026-08-19 23:56:57,825 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Google's revenue in 2023 was $307.4 billion.

Ask a question that contains just one metadata value:

In [23]:
answer_question("How important is diversity at Facebook?")

2026-08-19 23:57:00,501 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-08-19 23:57:01,344 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 2/2 [00:10<00:00,  5.40s/it]
2026-08-19 23:57:13,370 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Diversity is important at Facebook. The company is committed to creating a diverse and inclusive workplace, focusing on recruiting, retaining, and developing talent from underrepresented groups. They have specific goals to increase the representation of these groups, particularly in leadership positions. Efforts such as the Diverse Slate Approach in recruitment, diversity and inclusion training, and various community summits highlight the emphasis placed on diversity. Additionally, Facebook aims to have 50% of its workforce made up of underrepresented populations by 2024, underscoring the significance of diversity in their strategic initiatives.

Ask a question involving a company not represented in the database:

In [24]:
answer_question("What was Twitter's revenue in 2022?")

2026-08-19 23:57:16,347 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-08-19 23:57:17,354 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


I don't know.
